Yasmin Mohamed

# MediMatch Search Engine Design

**Section 1 - Installing and Importing Necessary Libraries**: This phase puts in place the Python environment using Python Terrier (PyTerrier), a complete toolkit for building information retrieval systems, and Pandas, a library for statistics manipulation and evaluation.

**Section 2 - Creating our Dataset**: It shows the process of collecting and organizing the health-associated information, which incorporates statistics about ddiseases, symptoms, and possible precautions, from variable sources for the foundational dataset for the MediMatch Search Engine.

**Section 3 - Data Preprocessing**: This component elaborates the preprocessing of the  dataset to make sure it has the correct format for indexing and retrieval.

**Section 4 - Enriching Results to Provide Additional Information**: This describes how the search engine's results are enriched with statistics like certain disorder descriptions and  precautions.

**Section 5 - Demonstration of MediMatch**: This final segment showcases the MediMatch Search Engine in action, demonstrating how customers can input signs and receive relevant disorder matches  illustrating the engine's functionality to resource information retrieval.

# Section 1 - Installing and Importing Necessary Libraries


**Python Terrier (Pyterrier)**



In [ ]:
!pip install python-terrier
import pyterrier as pt

pt.init()

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.7 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=1b4ced4254fb3823388d1811658ef45a754065de252f8dd354eb048275667907
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  

https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_1807/3859235080.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


**Pandas**

In [ ]:
!pip install pandas
import pandas as pd

# Section 2 : Creating our Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
file_path = '/content/drive/My Drive/HealthSearchEngineData/dataset.csv'

df = pd.read_csv(file_path)

In [ ]:
df_descriptions = pd.read_csv('/content/drive/My Drive/HealthSearchEngineData/disease_description.csv')
df_precautions = pd.read_csv('/content/drive/My Drive/HealthSearchEngineData/disease_precaution.csv')
df_severity = pd.read_csv('/content/drive/My Drive/HealthSearchEngineData/symptom_severity.csv')
df_training = pd.read_csv('/content/drive/My Drive/HealthSearchEngineData/Training.csv')
df_testing = pd.read_csv('/content/drive/My Drive/HealthSearchEngineData/Testing.csv')


# Section 3 - Data preprocessing

In [ ]:
# List of symptom columns in dataset
symptom_columns = [col for col in df.columns if 'Symptom_' in col]


df['text'] = df[symptom_columns].apply(lambda x: ' '.join(x.dropna()), axis=1)

df['docno'] = df.index.astype(str)

df[['text', 'docno']].head()

,text,docno
0,muscle_wasting patches_in_throat high_fever...,0
1,patches_in_throat high_fever extra_marital_...,1
2,muscle_wasting high_fever extra_marital_con...,2
3,muscle_wasting patches_in_throat extra_mari...,3
4,muscle_wasting patches_in_throat high_fever,4


**Indexing using Pyterrier**

In [ ]:
from pyterrier.index import IterDictIndexer

index_path = "/content/drive/My Drive/HealthSearchEngineData/my_index"

indexer = IterDictIndexer(index_path, overwrite=True)
index_ref = indexer.index(df[['docno', 'text']].to_dict(orient='records'))




/tmp/ipykernel_1807/3381582836.py:5: DeprecationWarning: Call to deprecated class IterDictIndexer. (use pt.terrier.IterDictIndexer() instead) -- Deprecated since version 0.11.0.
  indexer = IterDictIndexer(index_path, overwrite=True)


**Retrival and ranking**

In [ ]:
from pyterrier import BatchRetrieve


index = pt.IndexFactory.of(index_ref)

BM25_retriever = BatchRetrieve(index, wmodel="BM25")


query = "vomiting high_fever headache"
results = BM25_retriever.search(query)
print(results)



/tmp/ipykernel_1807/1828995827.py:6: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  BM25_retriever = BatchRetrieve(index, wmodel="BM25")


    qid  docid docno  rank     score                         query
0     1    248   248     0  3.369551  vomiting high_fever headache
1     1    249   249     1  3.224864  vomiting high_fever headache
2     1    254   254     2  3.092091  vomiting high_fever headache
3     1    258   258     3  3.092091  vomiting high_fever headache
4     1    251   251     4  2.969819  vomiting high_fever headache
..   ..    ...   ...   ...       ...                           ...
149   1    284   284   149  0.542955  vomiting high_fever headache
150   1    286   286   150  0.542955  vomiting high_fever headache
151   1    288   288   151  0.542955  vomiting high_fever headache
152   1    289   289   152  0.542955  vomiting high_fever headache
153   1    282   282   153  0.530412  vomiting high_fever headache

[154 rows x 6 columns]


# Section 4 - Enriching Results to  provide additional information


In [ ]:
print(df_training.columns)


Index(['abdominal_pain', 'abnormal_menstruation', 'acidity',
       'acute_liver_failure', 'altered_sensorium', 'anxiety', 'back_pain',
       'belly_pain', 'blackheads', 'bladder_discomfort',
       ...
       'watering_from_eyes', 'weakness_in_limbs', 'weakness_of_one_body_side',
       'weight_gain', 'weight_loss', 'yellow_crust_ooze', 'yellow_urine',
       'yellowing_of_eyes', 'yellowish_skin', 'prognosis'],
      dtype='object', length=135)


In [ ]:
results['docno'] = results['docno'].astype(int)


disease_mapping = df['Disease'].to_dict()


results['Disease'] = results['docno'].map(disease_mapping)


print(results[['docno', 'Disease']])


     docno                        Disease
0      248   Paralysis (brain hemorrhage)
1      249   Paralysis (brain hemorrhage)
2      254  Paroxysmal Positional Vertigo
3      258  Paroxysmal Positional Vertigo
4      251   Paralysis (brain hemorrhage)
..     ...                            ...
149    284                   Tuberculosis
150    286                   Tuberculosis
151    288                   Tuberculosis
152    289                   Tuberculosis
153    282                   Tuberculosis

[154 rows x 2 columns]


In [ ]:
description_mapping = df_descriptions.set_index('Disease')['Symptom_Description'].to_dict()
results['Description'] = results['Disease'].map(description_mapping)


for i in range(4):
    precaution_col = f'Symptom_precaution_{i}'
    precaution_mapping = df_precautions.set_index('Disease')[precaution_col].to_dict()
    results[precaution_col] = results['Disease'].map(precaution_mapping)

print(results[['docno', 'Disease', 'Description', 'Symptom_precaution_0', 'Symptom_precaution_1', 'Symptom_precaution_2', 'Symptom_precaution_3']])


     docno                        Disease  \
0      248   Paralysis (brain hemorrhage)   
1      249   Paralysis (brain hemorrhage)   
2      254  Paroxysmal Positional Vertigo   
3      258  Paroxysmal Positional Vertigo   
4      251   Paralysis (brain hemorrhage)   
..     ...                            ...   
149    284                   Tuberculosis   
150    286                   Tuberculosis   
151    288                   Tuberculosis   
152    289                   Tuberculosis   
153    282                   Tuberculosis   

                                           Description Symptom_precaution_0  \
0                                                  NaN              massage   
1                                                  NaN              massage   
2    Benign paroxysmal positional vertigo (BPPV) is...             lie down   
3    Benign paroxysmal positional vertigo (BPPV) is...             lie down   
4                                                  NaN          

In [ ]:
print(df_descriptions.head())


               Disease                                Symptom_Description
0                 AIDS  Acquired immunodeficiency syndrome (AIDS) is a...
1                 Acne  Acne vulgaris is the formation of comedones, p...
2  Alcoholic hepatitis  Alcoholic hepatitis is a diseased, inflammator...
3              Allergy  An allergy is an immune system response to a f...
4            Arthritis  Arthritis is the swelling and tenderness of on...


In [ ]:
import pandas as pd
import pyterrier as pt
if not pt.started():
    pt.init()

/tmp/ipykernel_1807/3658901313.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [ ]:
 print(df_precautions.columns)


Index(['Disease', 'Symptom_precaution_0', 'Symptom_precaution_1',
       'Symptom_precaution_2', 'Symptom_precaution_3'],
      dtype='object')


In [ ]:
precaution_columns = [col for col in df_precautions.columns if col.startswith('Precaution')]
for i, col in enumerate(precaution_columns, start=0):
    precaution_mapping = df_precautions.set_index('Disease')[col].to_dict()
    results[f'Symptom_precaution_{i}'] = results['Disease'].map(precaution_mapping)


In [ ]:
for i in range(4):
    precaution_col = f'Symptom_precaution_{i}'
    precaution_mapping = df_precautions.set_index('Disease')[precaution_col].to_dict()
    results[precaution_col] = results['Disease'].map(precaution_mapping)

print(results[['Disease', 'Symptom_precaution_0', 'Symptom_precaution_1', 'Symptom_precaution_2', 'Symptom_precaution_3']].head())


                         Disease Symptom_precaution_0  \
0   Paralysis (brain hemorrhage)              massage   
1   Paralysis (brain hemorrhage)              massage   
2  Paroxysmal Positional Vertigo             lie down   
3  Paroxysmal Positional Vertigo             lie down   
4   Paralysis (brain hemorrhage)              massage   

               Symptom_precaution_1        Symptom_precaution_2  \
0                       eat healthy                    exercise   
1                       eat healthy                    exercise   
2  avoid sudden changes in the body  avoid abrupt head movement   
3  avoid sudden changes in the body  avoid abrupt head movement   
4                       eat healthy                    exercise   

  Symptom_precaution_3  
0       consult doctor  
1       consult doctor  
2                relax  
3                relax  
4       consult doctor  


In [ ]:
print(df_descriptions.columns)


Index(['Disease', 'Symptom_Description'], dtype='object')


In [ ]:
description_mapping = df_descriptions.set_index('Disease')['Symptom_Description'].to_dict()
results['Description'] = results['Disease'].map(description_mapping)


print(results[['Disease', 'Description', 'Symptom_precaution_0', 'Symptom_precaution_1', 'Symptom_precaution_2', 'Symptom_precaution_3']].head())


                         Disease  \
0   Paralysis (brain hemorrhage)   
1   Paralysis (brain hemorrhage)   
2  Paroxysmal Positional Vertigo   
3  Paroxysmal Positional Vertigo   
4   Paralysis (brain hemorrhage)   

                                         Description Symptom_precaution_0  \
0                                                NaN              massage   
1                                                NaN              massage   
2  Benign paroxysmal positional vertigo (BPPV) is...             lie down   
3  Benign paroxysmal positional vertigo (BPPV) is...             lie down   
4                                                NaN              massage   

               Symptom_precaution_1        Symptom_precaution_2  \
0                       eat healthy                    exercise   
1                       eat healthy                    exercise   
2  avoid sudden changes in the body  avoid abrupt head movement   
3  avoid sudden changes in the body  avoid abrupt he

# Section 5 - Demonstration of MediMatch

In [ ]:
index = pt.IndexFactory.of('/content/drive/My Drive/HealthSearchEngineData/my_index/data.properties')
def search_and_display(query):

    BM25_retriever = pt.BatchRetrieve(index, wmodel="BM25")
    results = BM25_retriever.search(query)


    results['docno'] = results['docno'].astype(int)

    disease_mapping = df['Disease'].to_dict()
    results['Disease'] = results['docno'].map(disease_mapping)

    description_mapping = df_descriptions.set_index('Disease')['Symptom_Description'].to_dict()
    results['Description'] = results['Disease'].map(description_mapping)

    for i in range(4):
        precaution_col = f'Symptom_precaution_{i}'
        precaution_mapping = df_precautions.set_index('Disease')[precaution_col].to_dict()
        results[precaution_col] = results['Disease'].map(precaution_mapping)

    display_columns = ['docno', 'Disease', 'Description', 'Symptom_precaution_0', 'Symptom_precaution_1', 'Symptom_precaution_2', 'Symptom_precaution_3']
    display(results[display_columns].head(10))

query = input("Enter symptom: ")
search_and_display(query)


Enter symptom: headache


/tmp/ipykernel_1807/2027004861.py:4: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  BM25_retriever = pt.BatchRetrieve(index, wmodel="BM25")


,docno,Disease,Description,Symptom_precaution_0,Symptom_precaution_1,Symptom_precaution_2,Symptom_precaution_3
0,248,Paralysis (brain hemorrhage),NaN,massage,eat healthy,exercise,consult doctor
1,249,Paralysis (brain hemorrhage),NaN,massage,eat healthy,exercise,consult doctor
2,178,Hypertension,NaN,meditation,salt baths,reduce stress,get proper sleep
3,179,Hypertension,NaN,meditation,salt baths,reduce stress,get proper sleep
4,180,Hypertension,NaN,meditation,salt baths,reduce stress,get proper sleep
5,250,Paralysis (brain hemorrhage),NaN,massage,eat healthy,exercise,consult doctor
6,254,Paroxysmal Positional Vertigo,Benign paroxysmal positional vertigo (BPPV) is...,lie down,avoid sudden changes in the body,avoid abrupt head movement,relax
7,258,Paroxysmal Positional Vertigo,Benign paroxysmal positional vertigo (BPPV) is...,lie down,avoid sudden changes in the body,avoid abrupt head movement,relax
8,177,Hypertension,NaN,meditation,salt baths,reduce stress,get proper sleep
9,251,Paralysis (brain hemorrhage),NaN,massage,eat healthy,exercise,consult doctor
